[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/benchmarks/sampling_robustness.ipynb)

# MISDA — robustness to clean sampling

This notebook is a presentation front end for the reproducible sampling-robustness runner. It varies only the clean sample; observation noise remains `sigma=0`.


In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), None)
target = f"{repo_root}[benchmarks]" if repo_root is not None else "misda[benchmarks] @ git+https://github.com/monacofj/misda.git@main"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import pandas as pd
from misda.benchmarks.validation import CONTROLLED_PROBLEM_IDS, SAMPLING_REPLICATE_SEEDS, run_sampling_robustness

N = 300
MISDA_SEED = 123
PROBLEM_IDS = CONTROLLED_PROBLEM_IDS
REPLICATE_SEEDS = SAMPLING_REPLICATE_SEEDS


## Sampling protocol

Each replicate draws a new clean sample from the same theoretical problem. MISDA keeps a fixed internal seed, so the reported variation is attributable to finite-sample variability. Cases 12 and 13 remain known adversarial limitations.


In [ ]:
sampling_artifact = run_sampling_robustness(
    n=N, misda_seed=MISDA_SEED, problem_ids=PROBLEM_IDS, replicate_seeds=REPLICATE_SEEDS
)
sampling = pd.DataFrame(sampling_artifact["records"])
sampling_summary = pd.DataFrame(sampling_artifact["summary"])
sampling_summary_display = sampling_summary.copy()
sampling_summary_display["selected_unit_recovery"] = (
    sampling_summary_display["selected_unit_recovery"]
    .astype(object)
    .where(
        sampling_summary_display["selected_unit_recovery"].notna(),
        "N/A — structural units not uniquely declared",
    )
)
sampling_summary_display


## Interpretation

`latent_recovery` and `structural_recovery` are finite-replicate exact-recovery proportions, not fitted probabilities. For Cases 12 and 13, diagnostic rates are more informative than exact recovery because those cases intentionally exercise known limitations.

`selected_unit_recovery` is shown as `N/A — structural units not uniquely declared` when the benchmark intentionally does not declare a unique structural-unit partition. This is an unavailable comparison, not a failed or undefined numerical calculation.
